# Machine Learning Fundamentals
## Unit 07 — End-to-End Preprocessing with Pipelines

**Stage 2: Machine Learning | Beginner level | English B2**  
**Business scenario:** Melbourne house-price estimation  
**Environment:** Google Colab / Jupyter Notebook  
**Libraries:** Pandas, NumPy, Matplotlib, scikit-learn

---

## Business question

A real dataset often contains missing values and a mix of numeric and categorical features. If preprocessing is performed manually, it is easy to apply inconsistent transformations or accidentally use information from validation data.

In this lesson, we will build a repeatable workflow that handles preprocessing and model training as one object. We will use `ColumnTransformer` to process different column types and `Pipeline` to connect preprocessing to a Random Forest model.

## Learning outcomes

By the end of this notebook, you should be able to:

- Explain why preprocessing should be part of the model workflow.
- Separate numeric and categorical columns.
- Build different preprocessing steps with `ColumnTransformer`.
- Impute missing values and one-hot encode categories.
- Combine preprocessing and a model with `Pipeline`.
- Evaluate the full workflow using validation MAE.
- Explain how pipelines help reduce inconsistent preprocessing and data leakage.
- Use the fitted pipeline to predict a new example.

## Lesson roadmap

1. What a pipeline does
2. Prepare the dataset
3. Build numeric and categorical preprocessing
4. Add a Random Forest model
5. Fit and evaluate the complete pipeline
6. Inspect transformed features and make a new prediction
7. Practical challenge and readiness checklist

**Dataset:** `melb_data.csv`, containing historical Melbourne housing records. Upload the file to Google Colab before running the data-loading cell. The model is for learning purposes, not current property valuation.

---
# 1. Introduction

Pipelines are a simple way to keep your data preprocessing and modeling code organized. Specifically, a pipeline bundles preprocessing and modeling steps so you can use the whole bundle as if it were a single step.

Many data scientists build models without pipelines, but pipelines have important benefits:

- **Cleaner code:** Accounting for data at each step of preprocessing can get messy. With a pipeline, you don't need to manually keep track of every transformed version of the training and validation data.
- **Fewer bugs:** There are fewer opportunities to apply a step incorrectly or forget a preprocessing step.
- **Easier to productionize:** A single pipeline object can be saved and used to make predictions on new data, using the same preprocessing steps that were fitted during training.
- **Safer model validation:** When preprocessing is fitted inside the pipeline, it is easier to avoid data leakage during validation and cross-validation.


## 1.1 What Is a Pipeline?

Think of a pipeline as a sequence of connected steps:

**Raw features → Preprocessing → Model → Prediction**

For this housing example, preprocessing needs to handle two different kinds of data:

- **Numerical features:** fill in missing values.
- **Categorical features:** fill in missing values, then convert categories into numerical columns using one-hot encoding.

A `ColumnTransformer` lets us apply different preprocessing steps to different groups of columns. A `Pipeline` then connects the preprocessing step to the model. This is a standard pattern in scikit-learn. <https://scikit-learn.org/stable/modules/compose.html>


---
# 2. Prepare the Data

We continue with the Melbourne Housing dataset used in previous units. We separate the target (`Price`) from the input features, then split the rows before learning any preprocessing values.

We select categorical columns with fewer than 10 unique values for this introductory lesson. This is a simple teaching rule, not a universal feature-selection method.

**Why split first?** Any learned preprocessing step—such as an imputer—should learn its values from training data only. The validation set is reserved for evaluating the fitted workflow.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Read the data
melbourne_file_path = 'melb_data.csv'
data = pd.read_csv(melbourne_file_path)

# Separate target from predictors
y = data.Price
X = data.drop(['Price'], axis=1)

# Divide data into training and validation subsets
X_train_full, X_valid_full, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0
)

# Select categorical columns with relatively low cardinality
categorical_cols = [
    col for col in X_train_full.columns
    if X_train_full[col].nunique() < 10
    and X_train_full[col].dtype == 'object'
]

# Select numerical columns
numerical_cols = [
    col for col in X_train_full.columns
    if X_train_full[col].dtype in ['int64', 'float64']
]

# Keep selected columns only
my_cols = categorical_cols + numerical_cols
X_train = X_train_full[my_cols].copy()
X_valid = X_valid_full[my_cols].copy()

print("Categorical columns:", categorical_cols)
print("Number of numerical columns:", len(numerical_cols))
print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)

print("\nMissing values in selected training features:")
display(X_train.isna().sum().loc[lambda counts: counts > 0].sort_values(ascending=False))


## 2.1 Inspect the Training Data

We take a peek at the training data with `head()`. Notice that the data contains both categorical values and columns with missing values. A pipeline can handle both types of preprocessing.


In [ ]:
X_train.head()

---
# 3. Step 1: Define Preprocessing Steps

We use `ColumnTransformer` to apply different transformations to numeric and categorical columns.

- **Numeric features:** fill missing values with the median of each training column. The median is less affected by extreme values than the mean.
- **Categorical features:** fill missing values with the most frequent category, then apply one-hot encoding.
- **Unknown categories:** `handle_unknown='ignore'` prevents an error if validation or future data contains a category that was not present during fitting.

The imputation values and category mapping are learned when the pipeline is fitted on training data.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Numeric preprocessing: learn each column's median from training data
numerical_transformer = SimpleImputer(strategy="median")

# Categorical preprocessing: fill missing categories, then encode them
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Apply the appropriate transformation to each column group
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ],
    remainder="drop"
)

print(f"Numeric features: {len(numerical_cols)}")
print(f"Categorical features: {len(categorical_cols)}")


## 3.1 Why are there two preprocessing paths?

Numeric and categorical columns represent different kinds of information, so they need different preprocessing steps.

- A missing numeric value can be replaced with a statistic such as the median.
- A missing category can be replaced with the most frequent category.
- One-hot encoding creates indicator columns for categories instead of assigning an arbitrary numeric ranking to them.

`ColumnTransformer` applies these paths separately and combines their outputs into one feature matrix for the model.

---
# 4. Step 2: Define the Model

Next, we define a random forest model using the familiar `RandomForestRegressor` class.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=0,
    n_jobs=-1
)

The `RandomForestRegressor` predicts a numerical value, such as a house price. The `random_state` setting helps make the results reproducible when you run the notebook again.


---
# 5. Step 3: Create and Evaluate the Complete Pipeline

The `Pipeline` combines preprocessing and model training into one repeatable workflow.

When we call `fit()`:
1. The preprocessor learns imputation values and category mappings from the training data.
2. The training features are transformed.
3. The Random Forest is fitted on the transformed features.

When we call `predict()` with validation data, the pipeline applies the same learned transformations before generating predictions.

In [ ]:
from sklearn.metrics import mean_absolute_error
import numpy as np
import matplotlib.pyplot as plt

my_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

# Fit the entire workflow on training data only
my_pipeline.fit(X_train, y_train)

# The pipeline applies its fitted preprocessing automatically
preds = my_pipeline.predict(X_valid)
score = mean_absolute_error(y_valid, preds)

print(f"Validation MAE: AUD {score:,.0f}")

# Compare with a simple baseline that always predicts the training median
baseline_prediction = y_train.median()
baseline_preds = np.full(shape=len(y_valid), fill_value=baseline_prediction)
baseline_mae = mean_absolute_error(y_valid, baseline_preds)

comparison = pd.DataFrame({
    "Model": ["Training-median baseline", "Preprocessing + Random Forest pipeline"],
    "Validation MAE (AUD)": [baseline_mae, score]
}).sort_values("Validation MAE (AUD)")

display(comparison.style.format({"Validation MAE (AUD)": "AUD {:,.0f}"}))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(comparison["Model"], comparison["Validation MAE (AUD)"])
ax.set_title("Validation MAE: Baseline vs Pipeline")
ax.set_ylabel("Mean Absolute Error (AUD)")
ax.tick_params(axis="x", rotation=10)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


The original Kaggle example reports a validation MAE of approximately **AUD 160,679** under its specific data and preprocessing setup. Your score may differ because of dataset versions, library versions, selected features, and preprocessing choices.

The table above reports the result from your current run. Lower MAE indicates lower average absolute prediction error on this validation set. The baseline comparison provides context, but neither score guarantees performance on future properties.

## 5.1 What Happens When You Call `fit()`?

When you run:

```python
my_pipeline.fit(X_train, y_train)
```

the pipeline performs these operations in order:

1. Learns numerical replacement values from the training data.
2. Learns the most frequent category for categorical columns.
3. Learns the categories needed for one-hot encoding.
4. Transforms the training features.
5. Fits the random forest using the transformed features and target values.

When you call `my_pipeline.predict(X_valid)`, it reuses the learned preprocessing steps and then generates predictions. It does not learn new imputation values or category mappings from the validation set.


## 5.2 Inspect the transformed feature space

A `ColumnTransformer` may create more columns than the original dataset because one-hot encoding expands categorical features into indicator columns. The fitted preprocessor can show the output feature names, which helps us understand what the model receives.

In [ ]:
feature_names_out = my_pipeline.named_steps["preprocessor"].get_feature_names_out()

print(f"Original selected feature count: {len(my_cols)}")
print(f"Transformed feature count: {len(feature_names_out)}")
print("First 20 transformed feature names:")
print(feature_names_out[:20])

## 5.3 Predict for a new property

The main practical benefit is that the fitted pipeline can receive a DataFrame with the original input columns. It handles imputation and encoding before predicting, so we do not need to reproduce those steps manually.

The example below uses an existing validation row to demonstrate the interface. It is not a new independent evaluation.

In [ ]:
example_property = X_valid.iloc[[0]]
actual_price = y_valid.iloc[0]
predicted_price = my_pipeline.predict(example_property)[0]

display(example_property)
print(f"Actual price:    AUD {actual_price:,.0f}")
print(f"Predicted price: AUD {predicted_price:,.0f}")

## 5.4 Why Does This Help Prevent Data Leakage?

Data leakage occurs when information that should not be available during model training influences the training process or evaluation.

For example, if you calculate imputation values using the entire dataset before splitting it, information from validation rows influences preprocessing. This can make validation results less reliable.

A pipeline fitted on the training data keeps preprocessing steps within the training workflow. During validation or cross-validation, the preprocessing can be fitted separately within each training fold.


---
# 6. Conclusion and Practice

Pipelines are valuable for cleaning up machine learning code and avoiding errors, especially when a dataset needs several preprocessing steps.

In this lesson, you learned to:

- Separate numerical and categorical features.
- Impute missing values in both data types.
- Apply one-hot encoding to categorical features.
- Use `ColumnTransformer` to combine preprocessing paths.
- Use `Pipeline` to connect preprocessing and a Random Forest model.
- Evaluate the complete workflow with validation MAE.

## Your Turn

Use a pipeline in the next exercise to apply advanced data preprocessing techniques and improve your predictions.

**[Open Kaggle — Intermediate Machine Learning: Pipelines](https://www.kaggle.com/learn/intermediate-machine-learning/lesson/4)**

[Open the full Intermediate Machine Learning course](https://www.kaggle.com/learn/intermediate-machine-learning)

### Practical challenge

1. Run the complete pipeline and record its validation MAE.
2. Change numerical imputation from a constant to the median.
3. Try a different model, such as `DecisionTreeRegressor`, while keeping the preprocessing steps unchanged.
4. Compare the validation MAE values.
5. Explain why fitting preprocessing on training data only is important.

### Practice checklist

- [ ] I can explain what a pipeline does.
- [ ] I can use `ColumnTransformer`.
- [ ] I can define separate numerical and categorical preprocessing steps.
- [ ] I can combine preprocessing and a model with `Pipeline`.
- [ ] I can evaluate a complete pipeline using validation MAE.
- [ ] I can explain how pipelines help prevent data leakage.

## Key takeaway

**A pipeline connects preprocessing and modeling into one repeatable workflow, making machine learning code cleaner, safer, and easier to reuse.**


---
## References

- [Kaggle — Intermediate Machine Learning: Pipelines](https://www.kaggle.com/learn/intermediate-machine-learning/lesson/4)
- [scikit-learn — Pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html)
- [scikit-learn — SimpleImputer documentation](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html)
- [scikit-learn — OneHotEncoder documentation](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)
